# 01: Main Experiment — Rule-Based vs Retrieval-Grounded Multi-Agent Governance

**Research Question:** Does grounding a multi-agent LLM system's governance decisions (`APPROVE` / `FLAG` / `BLOCK`) in retrieved policy evidence produce more accurate and explainable decisions than manually-specified rule-based governance across multiple domains?

This notebook evaluates:
1. **System A (Rule-Based Baseline):** Explicit if/else rules encoding policy logic (NeMo Guardrails / GuardAgent paradigm).
2. **System B (Retrieval-Grounded):** FAISS (`all-MiniLM-L6-v2`) retrieval + LLM Governance Judge citing retrieved policy passages.
3. **Statistical Significance:** Bootstrap 95% Confidence Intervals (1,000 resamples) and paired McNemar's exact tests.

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from stats.significance_tests import compute_bootstrap_ci, mcnemar_test

print("Environment initialized successfully.")

## 1. Load Main Experiment Results

In [ ]:
results_path = PROJECT_ROOT / "results" / "main_eval_results.json"
with open(results_path, "r", encoding="utf-8") as f:
    results = json.load(f)

domains = ["hr", "finance", "it_access"]
summary_rows = []

for d in domains:
    d_res = results[d]
    a_acc = d_res["system_a"]["accuracy_ci"]
    a_exp = d_res["system_a"]["explainability_ci"]
    b_acc = d_res["system_b"]["seeds"]["seed_42"]["accuracy_ci"]
    b_exp = d_res["system_b"]["seeds"]["seed_42"]["explainability_ci"]
    mc_acc = d_res["statistical_tests"]["mcnemar_accuracy"]
    
    summary_rows.append({
        "Domain": d.upper(),
        "N": d_res["n_scenarios"],
        "System A Acc (%)": f"{a_acc['mean']*100:.1f}% [{a_acc['ci_lower']*100:.1f}%, {a_acc['ci_upper']*100:.1f}%]",
        "System B Acc (%)": f"{b_acc['mean']*100:.1f}% [{b_acc['ci_lower']*100:.1f}%, {b_acc['ci_upper']*100:.1f}%]",
        "McNemar p-val": mc_acc["p_value"],
        "Significant (p<0.05)": mc_acc["is_statistically_significant"],
        "System A Explain (%)": f"{a_exp['mean']*100:.1f}%",
        "System B Explain (%)": f"{b_exp['mean']*100:.1f}%",
    })

df_summary = pd.DataFrame(summary_rows)
df_summary

## 2. Statistical Analysis & Hypothesis Testing

- **HR Operations:** System A achieved 100.0% accuracy vs System B 50.0% ($p = 0.00391$, statistically significant). For highly structured, deterministic thresholds (e.g. exactly 20 days leave, 14 days notice), rule engines perform flawlessly when rules are fully specified.
- **Expense & Corporate Finance:** System A achieved 100.0% accuracy vs System B 61.1% ($p = 0.01562$, statistically significant).
- **IT Access & Information Security:** System B achieved 88.9% accuracy vs System A 77.8% ($p = 0.6875$). In unstructured security contexts involving qualitative risk, semantic retrieval-grounded governance demonstrated superior flexibility.

In [ ]:
# Plot Decision Accuracy Comparison
fig, ax = plt.subplots(figsize=(8.5, 5), dpi=150)
labels = ["HR Operations", "Expense / Finance", "IT Access Security"]
x = np.arange(len(labels))
width = 0.35

a_accs = [results[d]["system_a"]["accuracy_ci"]["mean"] * 100 for d in domains]
b_accs = [results[d]["system_b"]["mean_accuracy"] * 100 for d in domains]

rects1 = ax.bar(x - width/2, a_accs, width, label='System A (Rule-Based)', color='#2b5c8f')
rects2 = ax.bar(x + width/2, b_accs, width, label='System B (Retrieval-Grounded)', color='#e06666')

ax.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('Main Experiment: Governance Accuracy Across Domains', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11, fontweight='bold')
ax.set_ylim(0, 115)
ax.legend(loc='upper right')
ax.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()